# Generative AI & Prompt Engineering Internship Lab
**Author:** Adarsh Haldkar | B.Tech AIML, Shri Ram Institute of Technology (SRIT)

This notebook implements all 8 internship modules using the official `google-genai` SDK:
- **Modules 1–4:** LLM Tokenization, Model Comparison, Hallucination & Temperature Control, System Instructions, Few-Shot & CoT Prompting
- **Module 5:** Zero-Shot vs. Few-Shot Prompting, Roles, Contexts, Instructions & Constraints (RICCE Framework)
- **Module 6:** Deterministic Structured JSON Outputs (Pydantic) & Core NLP Workflows (Summarization, Classification, Generation)
- **Module 7:** AI-Assisted Coding/Debugging, Reusable Prompt Templates & Automated Evaluation Report
- **Module 8:** Capstone — Modular `AIProductivityToolkit` (Meeting Summarizer, Email Drafter, ATS Resume Tailor, Coding & Research Assistant)

In [ ]:
# Cell 1: Install Official Google GenAI SDK & Pydantic
!pip install -q -U google-genai pydantic

In [ ]:
# Cell 2: Initialize GenAI Client via Colab Secrets
import os
import json
import time
from string import Template
from typing import Literal
from pydantic import BaseModel, Field
from google import genai
from google.genai import types
from google.colab import userdata

try:
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    api_key = os.environ.get('GEMINI_API_KEY', 'YOUR_API_KEY_HERE')

client = genai.Client(api_key=api_key)
FAST_MODEL = 'gemini-3.5-flash-lite'
GENERAL_MODEL = 'gemini-4-hydrogen'

print('Client initialized successfully!')

## Part 1 (Modules 1–4): LLM Fundamentals, Tokenization & Model Comparison

In [ ]:
# Cell 3: Tokenization & Context Window Inspection (Module 1)
sample_prompts = [
    'Generative AI',
    'Transformers rely on multi-head self-attention to weigh token relationships across context windows.'
]

for text in sample_prompts:
    tokens = client.models.count_tokens(model=GENERAL_MODEL, contents=text)
    print(f'Text: "{text}"')
    print(f'Tokens: {tokens.total_tokens} | Words: {len(text.split())} | Ratio: {round(len(text.split())/tokens.total_tokens, 2)} words/token\n')

In [ ]:
# Cell 4: Model Tier Latency & Output Comparison (Module 4)
for model_id in [FAST_MODEL, GENERAL_MODEL]:
    start = time.time()
    res = client.models.generate_content(
        model=model_id,
        contents='Contrast Discriminative AI vs Generative AI in 3 concise bullets.'
    )
    elapsed = round(time.time() - start, 2)
    print(f'=== {model_id} ({elapsed}s) ===')
    print(res.text)
    print('-' * 50)

## Module 5: Zero-Shot vs. Few-Shot Prompting, Roles, Contexts, Instructions & Constraints

In [ ]:
# Cell 5: Zero-Shot vs. Few-Shot with Role, Context, Instructions & Constraints (RICCE)
ticket = 'Payment deducted twice via UPI for order #8821, refund still pending after 5 days.'

# 1. Zero-Shot Prompt
zero_shot = client.models.generate_content(
    model=FAST_MODEL,
    contents=f'Classify urgency and department for this support ticket: {ticket}'
)
print('=== Zero-Shot Output ===')
print(zero_shot.text.strip())

# 2. Few-Shot with Role, Context & Strict Output Constraints
few_shot_prompt = f"""Context: Fintech customer support triage system.
Task: Classify each ticket strictly as [PRIORITY | DEPARTMENT | ACTION].
Constraint: Output ONLY one single line matching the exact format. No extra commentary.

Ticket: "App crashes immediately when opening profile settings on Android 15."
Output: [HIGH | MOBILE_ENG | CREATE_BUG_TICKET]

Ticket: "How do I download my annual tax statement PDF?"
Output: [LOW | SUPPORT_OPS | SEND_HELP_LINK]

Ticket: "{ticket}"
Output:"""

few_shot = client.models.generate_content(
    model=FAST_MODEL,
    contents=few_shot_prompt,
    config=types.GenerateContentConfig(
        system_instruction='You are a strict automated ticket routing engine.',
        temperature=0.0
    )
)
print('\n=== Constrained Few-Shot Output ===')
print(few_shot.text.strip())

## Module 6: Structured Outputs (JSON) & Core NLP Workflows

In [ ]:
# Cell 6: Deterministic JSON Schemas for Classification, Summarization & Generation
class TicketAnalysis(BaseModel):
    category: Literal['Billing', 'Technical Bug', 'Account Access', 'Feature Request']
    sentiment: Literal['Positive', 'Neutral', 'Negative']
    urgency_score: int = Field(description='Score from 1 (lowest) to 5 (critical)')
    one_line_summary: str = Field(description='Concise 15-word max summary')
    suggested_reply: str = Field(description='Empathetic 2-sentence customer response')

nlp_res = client.models.generate_content(
    model=GENERAL_MODEL,
    contents=f'Analyze this customer message: "{ticket}"',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=TicketAnalysis,
        temperature=0.1
    )
)
print('=== Structured NLP JSON Output ===')
print(json.dumps(json.loads(nlp_res.text), indent=2))

## Module 7: AI-Assisted Coding/Debugging, Reusable Templates & Evaluation Report

In [ ]:
# Cell 7: AI-Assisted Code Debugging with Structured Root-Cause Output
buggy_code = '''
def calculate_moving_average(prices, window):
    averages = []
    for i in range(len(prices)):
        # Bug: slices past list bounds & divides by window even for partial windows
        avg = sum(prices[i:i+window]) / window
        averages.append(avg)
    return averages
'''

class CodeDebugReport(BaseModel):
    bugs_found: list[str]
    fixed_code: str
    time_complexity: str

debug_res = client.models.generate_content(
    model=GENERAL_MODEL,
    contents=f'Debug and optimize this Python function for valid sliding windows only:\n{buggy_code}',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=CodeDebugReport,
        temperature=0.1
    )
)
debug_data = json.loads(debug_res.text)
print('Bugs Found:', debug_data['bugs_found'])
print('Complexity:', debug_data['time_complexity'])
print('\nFixed Code:\n' + debug_data['fixed_code'])

In [ ]:
# Cell 8: Reusable Prompt Templates & Automated Evaluation Report
class PromptTemplate:
    def __init__(self, name: str, system_role: str, template_str: str):
        self.name = name
        self.system_role = system_role
        self.template = Template(template_str)

    def render(self, **kwargs) -> str:
        return self.template.safe_substitute(**kwargs)

class JudgeScore(BaseModel):
    accuracy_score: int = Field(description='1 to 5')
    constraint_adherence: int = Field(description='1 to 5')
    critique: str

summarizer_tpl = PromptTemplate(
    name='exec_brief_v1',
    system_role='You are an executive analyst. Be concise and factual.',
    template_str='Summarize the following text in exactly $max_bullets bullet points under 15 words each:\n\n$text'
)

test_cases = [
    'RAG combines vector retrieval with LLMs to reduce hallucinations by grounding answers in external documents.',
    'Quantization reduces model weights from FP16 to INT4, cutting GPU VRAM usage by up to 70% with minimal accuracy drop.'
]

print('=== Prompt Evaluation Report ===')
for idx, tc in enumerate(test_cases, 1):
    prompt = summarizer_tpl.render(max_bullets=2, text=tc)
    t0 = time.time()
    out = client.models.generate_content(
        model=FAST_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(system_instruction=summarizer_tpl.system_role, temperature=0.1)
    ).text
    latency = round(time.time() - t0, 2)

    # LLM-as-a-Judge Evaluation
    judge = client.models.generate_content(
        model=GENERAL_MODEL,
        contents=f'Evaluate this output against the prompt.\nPrompt: {prompt}\nOutput: {out}',
        config=types.GenerateContentConfig(response_mime_type='application/json', response_schema=JudgeScore)
    )
    scores = json.loads(judge.text)
    print(f'Test #{idx} | Latency: {latency}s | Accuracy: {scores["accuracy_score"]}/5 | Constraints: {scores["constraint_adherence"]}/5')
    print(f'Critique: {scores["critique"]}\n')

## Module 8: Capstone — AI Productivity Toolkit

In [ ]:
# Cell 9: Modular AIProductivityToolkit Class (Summarizer, Email, Resume, Coding/Research)
class MeetingSummary(BaseModel):
    tldr: str
    key_decisions: list[str]
    action_items: list[str]

class ResumeTailorOutput(BaseModel):
    match_score_percent: int
    missing_keywords: list[str]
    tailored_bullet_points: list[str]

class AIProductivityToolkit:
    def __init__(self, genai_client):
        self.client = genai_client

    def summarize_meeting(self, transcript: str) -> dict:
        res = self.client.models.generate_content(
            model=FAST_MODEL,
            contents=f'Extract structured meeting minutes from this transcript:\n{transcript}',
            config=types.GenerateContentConfig(
                response_mime_type='application/json',
                response_schema=MeetingSummary,
                temperature=0.1
            )
        )
        return json.loads(res.text)

    def draft_email(self, recipient: str, goal: str, tone: str = 'Professional & Concise') -> str:
        res = self.client.models.generate_content(
            model=FAST_MODEL,
            contents=f'Recipient: {recipient}\nGoal: {goal}\nTone: {tone}\nWrite a subject line and email body under 120 words.',
            config=types.GenerateContentConfig(
                system_instruction='You are an executive communications assistant.',
                temperature=0.4
            )
        )
        return res.text

    def tailor_resume(self, candidate_profile: str, job_description: str) -> dict:
        res = self.client.models.generate_content(
            model=GENERAL_MODEL,
            contents=f'Candidate Profile:\n{candidate_profile}\n\nTarget Job Description:\n{job_description}',
            config=types.GenerateContentConfig(
                system_instruction='You are an ATS resume optimization specialist. Quantify impact and align keywords truthfully.',
                response_mime_type='application/json',
                response_schema=ResumeTailorOutput,
                temperature=0.2
            )
        )
        return json.loads(res.text)

    def research_and_code_assistant(self, topic_or_task: str) -> str:
        res = self.client.models.generate_content(
            model=GENERAL_MODEL,
            contents=topic_or_task,
            config=types.GenerateContentConfig(
                system_instruction='Provide: (1) Concise architectural explanation, (2) Clean production Python snippet, and (3) Key trade-offs.',
                temperature=0.2
            )
        )
        return res.text

# --- Demo Execution of All 4 Toolkit Workflows ---
toolkit = AIProductivityToolkit(client)

print('1. MEETING SUMMARIZER:')
print(json.dumps(toolkit.summarize_meeting(
    'Adarsh: We finalized the RAG pipeline using ChromaDB. Priya will benchmark latency by Friday. Deployment moves to Monday.'
), indent=2))

print('\n2. EMAIL DRAFTER:')
print(toolkit.draft_email(
    recipient='Internship Coordinator, SRIT',
    goal='Submit GitHub repository and report for the Generative AI & Prompt Engineering internship.'
))

print('\n3. ATS RESUME TAILOR:')
print(json.dumps(toolkit.tailor_resume(
    candidate_profile='B.Tech AIML student (2027), Python, PyTorch, Google GenAI SDK, RAG pipelines, hackathon finalist.',
    job_description='Seeking AI Engineering Intern with experience in LLM APIs, structured JSON outputs, and prompt evaluation.'
), indent=2))

print('\n4. CODING & RESEARCH ASSISTANT:')
print(toolkit.research_and_code_assistant('Implement cosine similarity in NumPy for semantic search embeddings.'))